# Actionability tester (with preprocessing)

Same structure as `function_tester.ipynb` — one cell per function, same test sentences.  
The difference: preprocessing (`clean_text` + language filter) runs first, so every function
sees text exactly as it would in the real pipeline.

## Data
Same `test_action.csv` from the project root.

In [2]:
# 1) Load test data — same loader as function_tester
from __future__ import annotations
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
TEST_PATH = PROJECT_ROOT / 'test_action.csv'

print('Project root:', PROJECT_ROOT)
print('Test CSV path:', TEST_PATH)

raw = TEST_PATH.read_text(encoding='utf-8', errors='ignore')

rows = []
for line in raw.splitlines():
    line = line.strip()
    if not line:
        continue
    if line.lower().startswith('text') and 'lang' in line.lower():
        continue
    if '\t' in line:
        parts = [p.strip() for p in line.split('\t') if p.strip()]
    else:
        parts = [p.strip() for p in re.split(r'\s{2,}', line) if p.strip()]
    if len(parts) < 2:
        continue
    text = parts[0].strip().strip('"')
    lang = parts[-1].strip()
    if lang not in {'en', 'es', 'pt'}:
        continue
    rows.append({'clean_text': text, 'language': lang})

df_raw = pd.DataFrame(rows)
print('Loaded rows:', len(df_raw))
print('Language distribution:', df_raw['language'].value_counts().to_dict())
display(df_raw)

Project root: c:\Users\lenka\Documents\CSS-SEM4\Flood\Model
Test CSV path: c:\Users\lenka\Documents\CSS-SEM4\Flood\Model\test_action.csv
Loaded rows: 9
Language distribution: {'en': 3, 'pt': 3, 'es': 3}


,clean_text,language
0,"The tunnel is flooded, take an alternative route",en
1,"The flood took place at 2am, there are 40 dead...",en
2,If you live in Amsterdam you should avoid leav...,en
3,"O túnel está inundado, utilize uma rota altern...",pt
4,"A inundação ocorreu às 2h, há 40 mortos, eles ...",pt
5,"Se você mora em Amsterdão, deve evitar sair de...",pt
6,"El túnel está inundado, tome una ruta alternativa",es
7,"La inundación ocurrió a las 2am, hay 40 person...",es
8,"Si vive en Ámsterdam, debe evitar salir de su ...",es


In [3]:
# 2) Imports from the pipeline
import sys
import importlib

sys.path.insert(0, str(PROJECT_ROOT))

config        = importlib.import_module('config.nlp_config')
preprocessing = importlib.import_module('nlp.preprocessing')
actionability = importlib.import_module('nlp.actionability')

print('Supported languages:', config.SUPPORTED_LANGUAGES)
print('SpaCy models:', config.SPACY_MODELS)
print('Keyword dict languages:', list(config.ACTIONABILITY_KEYWORDS.keys()))

Supported languages: ['en', 'es', 'pt']
SpaCy models: {'en': 'en_core_web_sm', 'es': 'es_core_news_sm', 'pt': 'pt_core_news_sm'}
Keyword dict languages: ['en', 'es', 'pt']


In [4]:
# 3) Preprocessing step 1 — clean_text()
# Strips HTML, zero-width chars, collapses whitespace.
# Same function called in run_preprocessing() on the real CSV.
clean_text = preprocessing.clean_text

df = df_raw.copy()
df['clean_text'] = df['clean_text'].apply(clean_text)

for i, r in df.iterrows():
    original  = df_raw.loc[i, 'clean_text']
    cleaned   = r['clean_text']
    changed   = ' <- changed' if cleaned != original else ''
    print(f'--- row {i} | lang={r["language"]} | len {len(original)}->{len(cleaned)}{changed} ---')
    print(cleaned)
    print()

--- row 0 | lang=en | len 48->48 ---
The tunnel is flooded, take an alternative route

--- row 1 | lang=en | len 72->72 ---
The flood took place at 2am, there are 40 dead people, they were trapped

--- row 2 | lang=en | len 112->112 ---
If you live in Amsterdam you should avoid leaving your house, instead close all your windows and check for leaks

--- row 3 | lang=pt | len 51->51 ---
O túnel está inundado, utilize uma rota alternativa

--- row 4 | lang=pt | len 60->60 ---
A inundação ocorreu às 2h, há 40 mortos, eles ficaram presos

--- row 5 | lang=pt | len 118->118 ---
Se você mora em Amsterdão, deve evitar sair de casa; em vez disso, feche todas as janelas e verifique se há vazamentos

--- row 6 | lang=es | len 49->49 ---
El túnel está inundado, tome una ruta alternativa

--- row 7 | lang=es | len 76->76 ---
La inundación ocurrió a las 2am, hay 40 personas muertas, quedaron atrapadas

--- row 8 | lang=es | len 122->122 ---
Si vive en Ámsterdam, debe evitar salir de su casa; en su l

In [5]:
# 4) Preprocessing step 2 — language filter
# Keeps only rows whose language is in config.SUPPORTED_LANGUAGES.
# On the real pipeline this follows the ISO 639-2 -> 639-1 mapping;
# test_action.csv already uses 639-1 codes so no mapping needed here.

before = len(df)
df = df[df['language'].isin(config.SUPPORTED_LANGUAGES)].copy().reset_index(drop=True)
print(f'Language filter: {before} -> {len(df)} rows')
print('Languages present:', df['language'].value_counts().to_dict())
display(df)

Language filter: 9 -> 9 rows
Languages present: {'en': 3, 'pt': 3, 'es': 3}


,clean_text,language
0,"The tunnel is flooded, take an alternative route",en
1,"The flood took place at 2am, there are 40 dead...",en
2,If you live in Amsterdam you should avoid leav...,en
3,"O túnel está inundado, utilize uma rota altern...",pt
4,"A inundação ocorreu às 2h, há 40 mortos, eles ...",pt
5,"Se você mora em Amsterdão, deve evitar sair de...",pt
6,"El túnel está inundado, tome una ruta alternativa",es
7,"La inundación ocurrió a las 2am, hay 40 person...",es
8,"Si vive en Ámsterdam, debe evitar salir de su ...",es


In [6]:
# 5) Function: _truncate_at_sentence(text)
truncate = actionability._truncate_at_sentence

for i, r in df.iterrows():
    original  = r['clean_text']
    truncated = truncate(original)
    print(f'--- row {i} | lang={r["language"]} | original_len={len(original)} | truncated_len={len(truncated)} ---')
    print(truncated)
    print()

--- row 0 | lang=en | original_len=48 | truncated_len=48 ---
The tunnel is flooded, take an alternative route

--- row 1 | lang=en | original_len=72 | truncated_len=72 ---
The flood took place at 2am, there are 40 dead people, they were trapped

--- row 2 | lang=en | original_len=112 | truncated_len=112 ---
If you live in Amsterdam you should avoid leaving your house, instead close all your windows and check for leaks

--- row 3 | lang=pt | original_len=51 | truncated_len=51 ---
O túnel está inundado, utilize uma rota alternativa

--- row 4 | lang=pt | original_len=60 | truncated_len=60 ---
A inundação ocorreu às 2h, há 40 mortos, eles ficaram presos

--- row 5 | lang=pt | original_len=118 | truncated_len=118 ---
Se você mora em Amsterdão, deve evitar sair de casa; em vez disso, feche todas as janelas e verifique se há vazamentos

--- row 6 | lang=es | original_len=49 | truncated_len=49 ---
El túnel está inundado, tome una ruta alternativa

--- row 7 | lang=es | original_len=76 | trunc

In [7]:
# 6) Function: _get_spacy(lang)
get_spacy = actionability._get_spacy

for lang in sorted(df['language'].unique().tolist()):
    nlp = get_spacy(lang)
    print(f'lang={lang} -> model={config.SPACY_MODELS.get(lang)} -> loaded={nlp is not None}')
    if nlp is not None:
        print('  pipe:', nlp.pipe_names)
    print()

spacy model en_core_web_sm not found — run: python -m spacy download en_core_web_sm


lang=en -> model=en_core_web_sm -> loaded=False



spacy model pt_core_news_sm not found — run: python -m spacy download pt_core_news_sm


lang=es -> model=es_core_news_sm -> loaded=True
  pipe: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

lang=pt -> model=pt_core_news_sm -> loaded=False



In [8]:
# 7) Function: score_actionability_keywords(text, lang)
score_kw = actionability.score_actionability_keywords

out = []
for i, r in df.iterrows():
    res = score_kw(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_kw = pd.DataFrame(out)
display(df_kw)

print('Score formula: total = (imp*0.35) + (short*0.30) + (long*0.15) + (spatial*0.20)')

,row,language,imperative_score,short_term_score,long_term_score,spatial_score,actionability_score
0,0,en,0,0,0,0,0.00
1,1,en,0,0,0,0,0.00
2,2,en,1,0,0,0,0.35
3,3,pt,0,0,0,0,0.00
4,4,pt,0,0,0,0,0.00
5,5,pt,2,0,0,0,0.70
6,6,es,0,0,0,0,0.00
7,7,es,0,0,0,0,0.00
8,8,es,1,0,0,0,0.35


Score formula: total = (imp*0.35) + (short*0.30) + (long*0.15) + (spatial*0.20)


In [9]:
# 8) Function: extract_srl_features(text, lang)
extract_srl = actionability.extract_srl_features

out = []
for i, r in df.iterrows():
    res = extract_srl(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_srl = pd.DataFrame(out)
display(df_srl)

print('Interpretation: srl_complete=1 iff (has_agent and has_action and has_location)')

,row,language,has_agent,has_action,has_location,srl_complete
0,0,en,0,0,0,0
1,1,en,0,0,0,0
2,2,en,0,0,0,0
3,3,pt,0,0,0,0
4,4,pt,0,0,0,0
5,5,pt,0,0,0,0
6,6,es,1,0,0,0
7,7,es,1,1,0,0
8,8,es,0,1,1,0


Interpretation: srl_complete=1 iff (has_agent and has_action and has_location)


In [10]:
# 9) Function: extract_named_entities(text, lang)
extract_ner = actionability.extract_named_entities

out = []
for i, r in df.iterrows():
    res = extract_ner(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_ner = pd.DataFrame(out)
display(df_ner)

,row,language,top_locations,top_orgs
0,0,en,[],[]
1,1,en,[],[]
2,2,en,[],[]
3,3,pt,[],[]
4,4,pt,[],[]
5,5,pt,[],[]
6,6,es,[],[]
7,7,es,[],[]
8,8,es,"[""Ámsterdam""]",[]


In [11]:
# 10) Function: count_verb_tenses(text, lang)
count_tenses = actionability.count_verb_tenses

out = []
for i, r in df.iterrows():
    res = count_tenses(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_tenses = pd.DataFrame(out)
display(df_tenses)

,row,language,past_tense,present_tense,future_tense
0,0,en,0,0,0
1,1,en,0,0,0
2,2,en,0,0,0
3,3,pt,0,0,0
4,4,pt,0,0,0
5,5,pt,0,0,0
6,6,es,0,2,0
7,7,es,2,1,0
8,8,es,0,4,0


In [ ]:
# 11) Past tense ratio — replaces temporal phase
# Temporal phase scoring was removed: pub_date ranges span decades in the pilot dataset,
# making before/during/after phases meaningless. The past-tense penalty on actionability_score
# serves the same purpose (past-heavy articles describe what happened, not what to do).

out = []
for i, r in df.iterrows():
    tenses = actionability.count_verb_tenses(r['clean_text'], r['language'])
    total_verbs = tenses['past_tense'] + tenses['present_tense'] + tenses['future_tense']
    ratio = tenses['past_tense'] / total_verbs if total_verbs > 0 else 0.0
    out.append({
        'row': i,
        'language': r['language'],
        **tenses,
        'total_verbs': total_verbs,
        'past_tense_ratio': round(ratio, 3),
    })

df_ratio = pd.DataFrame(out)
display(df_ratio)
print('past_tense_ratio drives a 0.15-weight penalty on actionability_score in run_actionability()')

In [ ]:
# 12) Function: run_actionability(df)
# Full pipeline on the preprocessed df — no temporal phase column needed.
run_actionability = actionability.run_actionability

df_enriched = run_actionability(df.copy())

cols_to_show = [
    'clean_text', 'language',
    'imperative_score', 'short_term_score', 'long_term_score', 'spatial_score',
    'past_tense_ratio', 'srl_complete', 'actionability_score',
]
cols_to_show = [c for c in cols_to_show if c in df_enriched.columns]

display(df_enriched[cols_to_show])